In [41]:
# %% 1) Imports & config
import os, glob, re, json
from typing import Dict, Optional, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# === Configure your paths here ===
DATA_DIR   = "/Users/alexpotter/AIoT-Lab/Data/Before_Retro_Path_Folder"
# EITHER a single absolute file path OR a pattern inside DATA_DIR:
FILE_GLOB  = "/Users/alexpotter/AIoT-Lab/Data/Before_Retro_Path_Folder/49643052_All_Files_Combined_341_1.csv"
# FILE_GLOB = "*.csv"  # <- uncomment to use all CSVs in the folder

OUTPUT_DIR = "/Users/alexpotter/AIoT-Lab/Datapreprocessing"
HORIZON    = 1  # predict next-step indoor temperature

os.makedirs(OUTPUT_DIR, exist_ok=True)

# %% 2) Columns (time vs features)
REQUIRED_TIME_COL = "Timestamp"
REQUIRED_FEATURE_COLS = [
    "Indoor_Temperature_C",
    "Indoor_RH_Percent",
    "Outdoor_Temp_C",
    "Dewpoint_Temp_C",
    "Solar_Radiation_DNI_W_m2",
    "HVAC_Energy_usage_KWh",
]

# %% 3) Helpers (normalize, synonyms, mapping, timestamp parsing, I/O)
def _normalize(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(s).strip().lower())

SYNONYMS = {
    "indoortemperaturec": ["indoortemperaturec","indoor_temp_c","indoortemp","indoortemperature","indoortemperature_c"],
    "indoorrhpercent": ["indoorrhpercent","indoor_rh_percent","indoorrh","relativehumidity","indoorhumidity"],
    "outdoortempc": ["outdoortempc","outdoor_temp_c","outdoortemp","outdoortemperature","outdoortemperature_c"],
    "dewpointtempc": ["dewpointtempc","dewpoint_temp_c","dewpoint","dewpointc"],
    # Support "Solar_Radiation_DNI_W_m2" and common fallbacks
    "solarradiationdniwm2": ["solar_radiation_dni_w_m2","dni","dni_w_m2","solar_radiation","solarradiation","solar_dni"],
    "hvacenergyusagekwh": ["hvacenergyusagekwh","hvac_energy_usage_kwh","hvac_kwh","hvacenergykwh"],
}

def _map_required_features(df: pd.DataFrame) -> Dict[str, str]:
    n2a = {_normalize(c): c for c in df.columns}
    mapping = {}
    for req in REQUIRED_FEATURE_COLS:
        key = _normalize(req)
        cands = SYNONYMS.get(key, [key])
        found = None
        for c in cands:
            if c in n2a:
                found = n2a[c]; break
        if found is None:
            for k, actual in n2a.items():
                if key in k:
                    found = actual; break
        if found is None:
            raise ValueError(f"Missing required feature '{req}'. Found: {list(df.columns)}")
        mapping[req] = found
    return mapping

def _parse_timestamp_series(ts_raw: pd.Series) -> pd.Series:
    """Parse strings like '8/2/2021 14:17'. Try US, then EU, then generic."""
    ts = pd.to_datetime(ts_raw, format="%m/%d/%Y %H:%M", errors="coerce")
    if ts.isna().mean() > 0.5:
        ts_alt = pd.to_datetime(ts_raw, format="%d/%m/%Y %H:%M", errors="coerce")
        if ts_alt.notna().sum() > ts.notna().sum():
            ts = ts_alt
    if ts.isna().all():
        ts = pd.to_datetime(ts_raw, errors="coerce", infer_datetime_format=True)
    return ts

def read_and_select(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    # Normalize headers (strip BOM/whitespace)
    fixed_cols = []
    for c in df.columns:
        cc = str(c).strip()
        if cc.startswith("\ufeff"):
            cc = cc.lstrip("\ufeff")
        fixed_cols.append(cc)
    df.columns = fixed_cols

    # Find timestamp column: exact 'Timestamp' -> fuzzy 'timestamp'
    norm_map = {_normalize(c): c for c in df.columns}
    if REQUIRED_TIME_COL in df.columns:
        ts_col = REQUIRED_TIME_COL
    else:
        ts_col = norm_map.get("timestamp")
        if ts_col is None:
            for n, actual in norm_map.items():
                if "timestamp" in n:
                    ts_col = actual; break
    if ts_col is None:
        raise ValueError(f"Could not find a Timestamp column. Got headers: {list(df.columns)[:12]} ...")

    ts = _parse_timestamp_series(df[ts_col])

    # Map & extract features
    fmap = _map_required_features(df)
    out = df[[fmap[c] for c in REQUIRED_FEATURE_COLS]].copy()
    out.columns = REQUIRED_FEATURE_COLS
    for c in REQUIRED_FEATURE_COLS:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out[REQUIRED_TIME_COL] = ts
    out["__source_file"] = os.path.basename(path)
    print(f"[OK] {os.path.basename(path)} | Timestamp parsed (non-NaT): {int(ts.notna().sum())}/{len(ts)}")
    return out

def combine_files(data_dir: str, file_glob: str) -> pd.DataFrame:
    """Supports absolute single-file path OR a pattern inside data_dir."""
    if os.path.isabs(file_glob) and os.path.isfile(file_glob):
        paths = [file_glob]
    else:
        paths = sorted(glob.glob(os.path.join(data_dir, file_glob)))

    if not paths:
        raise FileNotFoundError(f"No files found. data_dir='{data_dir}', file_glob='{file_glob}'")

    frames = [read_and_select(p) for p in paths]
    combined = pd.concat(frames, axis=0, ignore_index=True)
    # Drop rows where ALL six features are NaN (keep Timestamp even if NaT)
    combined = combined.dropna(how="all", subset=REQUIRED_FEATURE_COLS)
    return combined

# %% 4) Combine & inspect
combined = combine_files(DATA_DIR, FILE_GLOB)
assert REQUIRED_TIME_COL in combined.columns, "Timestamp missing in combined."

print("Per-file row counts:")
display(combined["__source_file"].value_counts())
print("\nMissing values (Timestamp + features):")
display(combined[[REQUIRED_TIME_COL] + REQUIRED_FEATURE_COLS].isna().sum())
print("\nPreview:")
display(combined.head(10))

# %% 5) Clean (clip/ffill/median) — FEATURES ONLY (never touch Timestamp)
bounds = {
    "Indoor_Temperature_C": (-30, 60),
    "Outdoor_Temp_C": (-40, 60),
    "Indoor_RH_Percent": (0, 100),
    "Dewpoint_Temp_C": (-40, 35),
    "Solar_Radiation_DNI_W_m2": (0, None),
    "HVAC_Energy_usage_KWh": (0, None),
}

def clip_with_report(df: pd.DataFrame, bounds: dict) -> pd.DataFrame:
    df = df.copy()
    for col, (lo, hi) in bounds.items():
        if lo is not None: df.loc[df[col] < lo, col] = lo
        if hi is not None: df.loc[df[col] > hi, col] = hi
    return df

cleaned = clip_with_report(combined, bounds)

# Sort by file + time if timestamps exist
if cleaned[REQUIRED_TIME_COL].notna().any():
    cleaned = cleaned.sort_values(["__source_file", REQUIRED_TIME_COL]).reset_index(drop=True)
else:
    cleaned = cleaned.sort_values(["__source_file"]).reset_index(drop=True)

# Forward-fill within file for FEATURES only, then median-fill leftovers
cleaned[REQUIRED_FEATURE_COLS] = cleaned.groupby("__source_file")[REQUIRED_FEATURE_COLS].ffill()
for col in REQUIRED_FEATURE_COLS:
    cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce").fillna(cleaned[col].median())

print("Missing values after cleaning:")
display(cleaned[[REQUIRED_TIME_COL] + REQUIRED_FEATURE_COLS].isna().sum())
display(cleaned.head(10))

# %% 6) Save your “big CSV”
combined_csv_path = os.path.join(OUTPUT_DIR, "combined_features.csv")
cleaned.to_csv(combined_csv_path, index=False)
print("Saved:", combined_csv_path)

# %% 7) Define make_target
def make_target(df: pd.DataFrame, horizon: int = 1) -> pd.DataFrame:
    """
    Create next-step target for Indoor_Temperature_C.
    Keeps Timestamp; adds 'target_Indoor_Temperature_C_next'; drops last `horizon` NaNs.
    """
    if horizon <= 0:
        raise ValueError("HORIZON must be a positive integer")
    if "Indoor_Temperature_C" not in df.columns:
        raise KeyError("Column 'Indoor_Temperature_C' not found.")
    df = df.copy()
    df["Indoor_Temperature_C"] = pd.to_numeric(df["Indoor_Temperature_C"], errors="coerce")
    df["target_Indoor_Temperature_C_next"] = df["Indoor_Temperature_C"].shift(-horizon)
    df = df.dropna(subset=["target_Indoor_Temperature_C_next"])
    return df

# %% 8) Build `data` (optionally sort by time)
data = make_target(cleaned.drop(columns=["__source_file"]), HORIZON)
if data[REQUIRED_TIME_COL].notna().any():
    data = data.sort_values([REQUIRED_TIME_COL]).reset_index(drop=True)
print("`data` rows:", len(data))
display(data.head(10))

# %% 9) Split + add simple time-derived features (Timestamp stays out of model)
def chronological_split(df: pd.DataFrame, train_ratio: float = 0.8):
    n = len(df); split_idx = int(n * train_ratio)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

train_df, test_df = chronological_split(data, train_ratio=0.8)

for df_ in (train_df, test_df):
    ts = pd.to_datetime(df_[REQUIRED_TIME_COL], errors="coerce")
    df_["Hour"] = ts.dt.hour
    df_["DayOfWeek"] = ts.dt.dayofweek
    df_["Month"] = ts.dt.month

feature_cols = REQUIRED_FEATURE_COLS + ["Hour", "DayOfWeek", "Month"]

X_train = train_df[feature_cols].apply(pd.to_numeric, errors="coerce")
y_train = train_df["target_Indoor_Temperature_C_next"].astype(float)
X_test  = test_df[feature_cols].apply(pd.to_numeric, errors="coerce")
y_test  = test_df["target_Indoor_Temperature_C_next"].astype(float)

print("Features used:", feature_cols)
print("Train/Test sizes:", len(X_train), len(X_test))
print("X_train dtypes:\n", X_train.dtypes)

[OK] 49643052_All_Files_Combined_341_1.csv | Timestamp parsed (non-NaT): 455397/455397
Per-file row counts:


__source_file
49643052_All_Files_Combined_341_1.csv    455397
Name: count, dtype: int64


Missing values (Timestamp + features):


Timestamp                       0
Indoor_Temperature_C            0
Indoor_RH_Percent               0
Outdoor_Temp_C              21499
Dewpoint_Temp_C             21499
Solar_Radiation_DNI_W_m2    21499
HVAC_Energy_usage_KWh       33649
dtype: int64


Preview:


...

Timestamp                   0
Indoor_Temperature_C        0
Indoor_RH_Percent           0
Outdoor_Temp_C              0
Dewpoint_Temp_C             0
Solar_Radiation_DNI_W_m2    0
HVAC_Energy_usage_KWh       0
dtype: int64

,Indoor_Temperature_C,Indoor_RH_Percent,Outdoor_Temp_C,Dewpoint_Temp_C,Solar_Radiation_DNI_W_m2,HVAC_Energy_usage_KWh,Timestamp,__source_file
0,23.333333,58.5,23.28,8.9645,865.0,0.000001,2021-08-02 14:17:00,49643052_All_Files_Combined_341_1.csv
1,23.233333,58.7,23.28,8.9460,868.0,0.000001,2021-08-02 14:18:00,49643052_All_Files_Combined_341_1.csv
2,23.133333,58.9,23.28,8.9275,867.0,0.000001,2021-08-02 14:19:00,49643052_All_Files_Combined_341_1.csv
3,23.033333,59.1,23.28,8.9090,860.0,0.000001,2021-08-02 14:20:00,49643052_All_Files_Combined_341_1.csv
4,22.933333,59.3,23.28,8.8905,854.0,0.000001,2021-08-02 14:21:00,49643052_All_Files_Combined_341_1.csv
5,22.833333,59.5,23.28,8.8720,847.0,0.000001,2021-08-02 14:22:00,49643052_All_Files_Combined_341_1.csv
6,22.733333,59.7,23.28,8.8535,842.0,0.000001,2021-08-02 14:23:00,49643052_All_Files_Combined_341_1.csv
7,22.633333,59.9,23.28,8.8350,844.0,0.000001,2021-08-02 14:24:00,49643052_All_Files_Combined_341_1.csv
8,22.533333,60.1,23.28,8.8165,842.0,0.000001,2021-08-02 14:25:00,49643052_All_Files_Combined_341_1.csv
9,22.433333,60.3,23.28,8.7980,840.0,0.000001,2021-08-02 14:26:00,49643052_All_Files_Combined_341_1.csv


Saved: /Users/alexpotter/AIoT-Lab/Datapreprocessing/combined_features.csv
`data` rows: 455396


,Indoor_Temperature_C,Indoor_RH_Percent,Outdoor_Temp_C,Dewpoint_Temp_C,Solar_Radiation_DNI_W_m2,HVAC_Energy_usage_KWh,Timestamp,target_Indoor_Temperature_C_next
0,23.333333,58.5,23.28,8.9645,865.0,0.000001,2021-08-02 14:17:00,23.233333
1,23.233333,58.7,23.28,8.9460,868.0,0.000001,2021-08-02 14:18:00,23.133333
2,23.133333,58.9,23.28,8.9275,867.0,0.000001,2021-08-02 14:19:00,23.033333
3,23.033333,59.1,23.28,8.9090,860.0,0.000001,2021-08-02 14:20:00,22.933333
4,22.933333,59.3,23.28,8.8905,854.0,0.000001,2021-08-02 14:21:00,22.833333
5,22.833333,59.5,23.28,8.8720,847.0,0.000001,2021-08-02 14:22:00,22.733333
6,22.733333,59.7,23.28,8.8535,842.0,0.000001,2021-08-02 14:23:00,22.633333
7,22.633333,59.9,23.28,8.8350,844.0,0.000001,2021-08-02 14:24:00,22.533333
8,22.533333,60.1,23.28,8.8165,842.0,0.000001,2021-08-02 14:25:00,22.433333
9,22.433333,60.3,23.28,8.7980,840.0,0.000001,2021-08-02 14:26:00,22.333333


Features used: ['Indoor_Temperature_C', 'Indoor_RH_Percent', 'Outdoor_Temp_C', 'Dewpoint_Temp_C', 'Solar_Radiation_DNI_W_m2', 'HVAC_Energy_usage_KWh', 'Hour', 'DayOfWeek', 'Month']
Train/Test sizes: 364316 91080
X_train dtypes:
 Indoor_Temperature_C        float64
Indoor_RH_Percent           float64
Outdoor_Temp_C              float64
Dewpoint_Temp_C             float64
Solar_Radiation_DNI_W_m2    float64
HVAC_Energy_usage_KWh       float64
Hour                          int32
DayOfWeek                     int32
Month                         int32
dtype: object
